# CUDA Lab 01



In [11]:
!nvidia-smi

Thu Oct  8 07:13:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             28W /   70W |     111MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
from numba import cuda
import os
device = cuda.get_current_device()
name = device.name.decode('utf-8') if isinstance(device.name, bytes) else device.name
print(f"Device Name: {name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


## Section 2: crossover experiment

In [13]:
import numpy as np
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [14]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup (JIT compile) so compilation time is not measured
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0
    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
results = {}
print(f"{'N':>12} | {'CPU ms':>12} | {'GPU total ms':>13} | {'GPU kernel ms':>14} | Winner")
print("-"*70)
for N in sizes:
    c, g, k = benchmark_run(N)
    results[N] = (c, g, k)
    winner = "n/a (CPU skipped)" if c != c else ("GPU" if g < c else "CPU")
    print(f"{N:>12,} | {c:>12.4f} | {g:>13.4f} | {k:>14.4f} | {winner}")

# crossover: first N where GPU total < CPU
crossover = None
for N in sizes:
    c, g, k = results[N]
    if c == c and g < c:
        crossover = N
        break
print("\nEmpirical crossover N* =", crossover)
for N in (100,):
    c, g, k = results[N]
    print(f"N=100: kernel-only is {k/g*100:.1f}% of GPU total -> {100-k/g*100:.1f}% is overhead (PCIe/allocation/launch)")

           N |       CPU ms |  GPU total ms |  GPU kernel ms | Winner
----------------------------------------------------------------------
         100 |       0.0583 |        0.6814 |         0.0660 | CPU
       1,000 |       0.2494 |        0.5160 |         0.0599 | CPU
      10,000 |       2.4222 |        0.5021 |         0.0582 | GPU
     100,000 |      23.8138 |        1.2382 |         0.1193 | GPU


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


   1,000,000 |     242.5911 |        4.6951 |         0.1111 | GPU
   5,000,000 |          nan |       12.4538 |         0.2693 | n/a (CPU skipped)
  10,000,000 |          nan |       24.8479 |         0.4393 | n/a (CPU skipped)

Empirical crossover N* = 10000
N=100: kernel-only is 9.7% of GPU total -> 90.3% is overhead (PCIe/allocation/launch)


## Section 3: missing guard

In [15]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0   # no bounds check on purpose

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
print("Launching defective kernel with 1024 threads on 1000-element array...")
try:
    defective_kernel[4, 256](d_arr)
    cuda.synchronize()
    print("Execution finished (or did it crash?). Checking host copy...")
    h_result = d_arr.copy_to_host()
    print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")
except Exception as e:
    print("EXCEPTION:", type(e).__name__, e)

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [16]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
total = blocks_per_grid * threads_per_block
log_b = cuda.device_array(total, dtype=np.int32)
log_t = cuda.device_array(total, dtype=np.int32)
log_g = cuda.device_array(total, dtype=np.int32)
log_a = cuda.device_array(total, dtype=np.int32)
inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()
hb, ht, hg, ha = (x.copy_to_host() for x in (log_b, log_t, log_g, log_a))
print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):
    status = "ACTIVE" if ha[i] == 1 else "IDLE (GUARDED)"
    print(f"{hg[i]:<10} | {hb[i]:<10} | {ht[i]:<10} | {status:<10}")
print(f"\nBlocks: {blocks_per_grid} | Threads launched: {total} | Active: {int(ha.sum())} | Idle: {total - int(ha.sum())}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)

Blocks: 5 | Threads launched: 40 | Active: 37 | Idle: 3


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


## Section 4: race condition

In [17]:
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
d_ones = cuda.to_device(np.ones(N, dtype=np.float32))
tpb = 256
bpg = (N + tpb - 1) // tpb

def run(kernel):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    kernel[bpg, tpb](d_ones, d_total)
    cuda.synchronize()
    return d_total.copy_to_host()[0]

run(naive_sum_kernel); run(atomic_sum_kernel)   # warmup (JIT)

print(f"{'Trial':<8}{'Target':>10}{'Naive output':>16}{'Lost updates':>16}")
for t in range(1, 6):
    v = run(naive_sum_kernel)
    print(f"Trial {t:<2}{50000.0:>10.1f}{v:>16.1f}{50000.0 - v:>16.1f}")

a = run(atomic_sum_kernel)
print(f"\nAtomic sum: {a}  -> exactly 50000.0? {'Yes' if a == 50000.0 else 'No'}")

def timeit(kernel, reps=20):
    ts = []
    for _ in range(reps):
        d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
        cuda.synchronize()
        t0 = time.perf_counter()
        kernel[bpg, tpb](d_ones, d_total)
        cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000.0)
    return float(np.median(ts))
print(f"Naive kernel: {timeit(naive_sum_kernel):.4f} ms | Atomic kernel: {timeit(atomic_sum_kernel):.4f} ms (median of 20)")

Trial       Target    Naive output    Lost updates
Trial 1    50000.0             2.0         49998.0
Trial 2    50000.0             2.0         49998.0
Trial 3    50000.0             2.0         49998.0
Trial 4    50000.0             2.0         49998.0
Trial 5    50000.0             2.0         49998.0

Atomic sum: 50000.0  -> exactly 50000.0? Yes
Naive kernel: 0.0994 ms | Atomic kernel: 0.2818 ms (median of 20)


## Section 5: 2D grid

In [18]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)
threads_2d = (16, 16)
blocks_2d = ((WIDTH + threads_2d[0] - 1) // threads_2d[0],
             (HEIGHT + threads_2d[1] - 1) // threads_2d[1])
radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Shape: {result_canvas.shape}. Center value: {result_canvas[512, 512]:.3f}")
print(f"Blocks X: {blocks_2d[0]} | Blocks Y: {blocks_2d[1]} | Total: {blocks_2d[0]*blocks_2d[1]}")
print(f"Value at (0,0): {result_canvas[0, 0]:.3f}")

Shape: (1024, 1024). Center value: 1.000
Blocks X: 64 | Blocks Y: 64 | Total: 4096
Value at (0,0): 0.000


## Section 6: checksum


In [19]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    n = cuda.get_current_device().name
    h.update(n if isinstance(n, bytes) else n.encode('utf-8'))
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230103201"
CROSSOVER_N = crossover
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 1CE07ECC327EF647


'1CE07ECC327EF647'